# Automated comparison template

**Contributors:** initial scaffold and local verification prepared with AI assistance. Team reviewers and substantive changes will be recorded after review. See [the assistance record](../docs/AI_USE.md).

**Not executed.** Follow the lecturer's installation instructions in a separate compatible Python environment, such as 3.11. Do not add PyCaret to the verified Python 3.12 baseline environment.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src' / 'mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import json
import pandas as pd
from pycaret.classification import ClassificationExperiment
from mushrooms import load_data, make_split
frame = load_data()
splits = make_split(frame)
train = frame.loc[splits['train']].copy()
validation = frame.loc[splits['validation']].copy()
train['class'] = train['class'].map({'e': 0, 'p': 1})
validation['class'] = validation['class'].map({'e': 0, 'p': 1})

## Compare within the training data

`test_data` receives the existing validation partition, so the reserved final test stays untouched. Cross-validation runs inside the training partition. Verify the exact preprocessing and class mapping before interpreting recall.

In [ ]:
experiment = ClassificationExperiment()
experiment.setup(data=train, test_data=validation, target='class', session_id=42,
                 fold=5, fold_strategy='stratifiedkfold', normalize=True, verbose=False)
best = experiment.compare_models(sort='Recall', n_select=3)
comparison = experiment.pull()
display(comparison)
comparison.to_csv(ROOT/'reports/pycaret_development_comparison.csv')

Explain how each shortlisted algorithm works, tune it on development folds, and export metrics. Do not select a model by recall alone: a classifier that always predicts poisonous has perfect recall but no useful discrimination. Compare precision, PR behavior and the confusion matrix too.